# 00 — Async API Fetching & Caching

**Skill focus:** Data Engineering (intern/co-op/new grad)

This notebook demonstrates production-grade API ingestion patterns:
- **aiohttp + asyncio** — concurrent HTTP requests
- **Semaphore(20)** — rate limiting to respect API fair use
- **Exponential backoff** — retry on 429/5xx (1s → 2s → 4s)
- **Cache-first** — idempotent pipeline; re-runs never re-fetch
- **Ingestion manifest** — per-record status, bytes, timestamp

In [ ]:
# Run once on Databricks; installs deps from pyproject.toml. Restart Python if imports fail.
from pathlib import Path
import sys
root = Path.cwd()
for _ in range(3):
    if (root / "pyproject.toml").exists():
        break
    root = root.parent
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(root), "-q"])

In [ ]:
from src.env import project_root, CACHE_ROOT
from src.ingestion import AsyncPokeAPIFetcher
from src.constants import POKEAPI_ENDPOINTS
import asyncio

## 1. Configure fetcher and cache paths

- **Local:** uses `data/cache/` (default)
- **Databricks:** pass `cache_root="/dbfs/FileStore/pokedata/cache"`

In [ ]:
fetcher = AsyncPokeAPIFetcher(
    cache_root=str(CACHE_ROOT),
    concurrency=20,
    max_retries=3,
)

print(f"Cache root: {fetcher.cache_root}")
print(f"Manifest:   {fetcher.manifest_path}")
print(f"Run ID:     {fetcher.run_id}")

## 2. Async fetch all endpoints

- Checks cache before each request (cache hit → skip)
- Semaphore caps 20 concurrent connections
- tqdm progress bar per endpoint

In [ ]:
DEMO_ENDPOINTS = {
    "pokemon": POKEAPI_ENDPOINTS["pokemon"],
    "pokemon-species": POKEAPI_ENDPOINTS["pokemon-species"],
    "type": POKEAPI_ENDPOINTS["type"],
    "move": POKEAPI_ENDPOINTS["move"],
}

results = await fetcher.fetch_all(DEMO_ENDPOINTS)

## 3. Post-fetch validation

Assert file counts match expected totals per endpoint.

In [ ]:
valid = fetcher.validate_cache(DEMO_ENDPOINTS)
print(f"\nValidation: {'PASS' if valid else 'WARN (check output above)'}")

## 4. Manifest summary

Total records, success/failure counts, and cache size.

In [ ]:
summary = fetcher.manifest_summary()
for k, v in summary.items():
    print(f"  {k}: {v}")

## Key findings

- **Cache-first:** Re-running the notebook uses cached files; no duplicate API calls
- **Rate limiting:** Semaphore(20) prevents overwhelming the API
- **Resilience:** Exponential backoff handles transient 429/5xx errors
- **Traceability:** Manifest records status, bytes, and timestamp per (endpoint, id)